# Search Providers Playground

Interactive sandbox to try every provider in `mcp/server/search/providers/`.  
Run **Setup** first, then execute whichever provider cell you like.

In [1]:
# ── Setup: path + env ──────────────────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

# This notebook lives at  mcp/server/search/providers_playground.ipynb
# Project root is 3 levels up: search/ → server/ → mcp/ → <ROOT>
_HERE = Path("__file__").resolve().parent  # mcp/server/search/
ROOT = _HERE.parents[2]                    # project root
SEARCH_ROOT = ROOT / "mcp" / "server"

for p in (str(ROOT), str(SEARCH_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(ROOT / ".env")

import os, pprint

def show(result):
    """Pretty-print the key fields of a WebSearchResponse."""
    print(f"provider : {result.provider}")
    print(f"model    : {result.model}")
    print(f"results  : {len(result.search_results)}")
    if result.answer:
        print(f"answer   :\n{result.answer[:400]}")
    for i, r in enumerate(result.search_results[:5], 1):
        print(f"  [{i}] {r.title}")
        print(f"      {r.url}")
        if r.snippet:
            print(f"      {r.snippet[:120]}")

print("Setup complete.")
print(f"  ROOT        = {ROOT}")
print(f"  SEARCH_ROOT = {SEARCH_ROOT}")


Setup complete.
  ROOT        = /Users/lscm/Project/PCS/HSCodeSearch
  SEARCH_ROOT = /Users/lscm/Project/PCS/HSCodeSearch/mcp/server


---
## 1. DuckDuckGo  
Free, no API key required.

In [3]:
from search.providers.duckduckgo import DuckDuckGoProvider

provider = DuckDuckGoProvider()
result = provider.search("HS code ", max_results=5)
show(result)

provider : duckduckgo
model    : duckduckgo
results  : 0


/Users/lscm/Project/PCS/HSCodeSearch/mcp/server/search/providers/duckduckgo.py:36: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS(proxy=self.proxy, timeout=timeout)


---
## 2. Serper (Google SERP)  
Requires `SERPER_API_KEY` in `.env`.

In [3]:
from search.providers.serper import SerperProvider

provider = SerperProvider(api_key=os.getenv("SERPER_API_KEY"))
result = provider.search("HS code Samsung Galaxy S8 smartphone", num=5)
show(result)

provider : serper
model    : serper-search
results  : 5
  [1] Global Buyers of Samsung Galaxy S8 And Hsn Code 8517 - Volza
      https://www.volza.com/p/samsung-galaxy-s8/hsn-code-8517/
      Global Import Trade data of Samsung Galaxy S8 And Hsn Code 8517 ; Date. 27-May-2025 ; HSN Code. 8517792100 ; Product. De
  [2] Turkey Imports – Samsung galaxy s8 plus ( HS Code 851713000000)
      https://eximtradedata.com/search/country-turkey/type-import/product-samsung-galaxy-s8-plus/hscode-851713000000
      Access 2025 Turkey import data for Samsung galaxy s8 plus ( HS Code 851713000000) with verified shipment details, import
  [3] Samsung Galaxy Imports Under Sub Chapter 8517 - Zauba
      https://www.zauba.com/import-SAMSUNG+GALAXY/hs-code-8517-hs-code.html
      Information and reports on Samsung Galaxy Imports Under Sub Chapter 8517 along with detailed shipment data, import price
  [4] Samsung s8 HS Code for Export & Import - Seair Exim Solutions
      https://www.seair.co.in/samsung-s8-h

---
## 3. Tavily AI Search  
Requires `TAVILY_API_KEY` in `.env`.

In [4]:
from search.providers.tavily import TavilyProvider

provider = TavilyProvider(api_key=os.getenv("TAVILY_API_KEY"))
result = provider.search("HS code for smartphone Samsung Galaxy S8", max_results=5)
show(result)

provider : tavily
model    : tavily-basic
results  : 5
answer   :
The HS code for the Samsung Galaxy S8 is 85171300. This code applies to smartphones. Additional specific codes like 8517792100 may also be used depending on the exact model and features.
  [1] Global Buyers of Samsung Galaxy S8 And Hsn Code 8517 - Volza
      https://www.volza.com/p/samsung-galaxy-s8/hsn-code-8517/
      Global Import Trade data of Samsung Galaxy S8 And Hsn Code 8517 ; Date. 27-May-2025 ; HSN Code. 8517792100 ; Product. De
  [2] Turkey Imports – Samsung galaxy s8 plus ( HS Code 851713000000)
      https://eximtradedata.com/search/country-turkey/type-import/product-samsung-galaxy-s8-plus/hscode-851713000000
      Access 2025 Turkey import data for Samsung galaxy s8 plus ( HS Code 851713000000) with verified shipment details, import
  [3] Samsung Smartphone Imports Under HS Code 85171290 - Zauba
      https://www.zauba.com/import-samsung+smartphone/hs-code-85171290-hs-code.html
      Samsung Smartphone wor

---
## 4. Perplexity AI  
Requires `PERPLEXITY_API_KEY` in `.env`.  
Returns an LLM-generated answer with citations.

In [2]:
from search.providers.perplexity import PerplexityProvider

provider = PerplexityProvider(api_key=os.getenv("PERPLEXITY_API_KEY"))
result = provider.search(
    "What is the HS code for Samsung Galaxy S8 smartphone? Provide the 6-digit code and chapter."
)
show(result)

provider : perplexity
model    : sonar
results  : 3
answer   :
**The 6-digit HS code for the Samsung Galaxy S8 smartphone is 851713.**

This code falls under **Chapter 85** (Electrical machinery and equipment and parts thereof; sound recorders and reproducers, television image and sound recorders and reproducers, and parts and accessories of such articles).[2]  
Smartphones like the Samsung Galaxy S8 (and S8 Plus) are classified under heading 8517 (Telephone 
  [1] Samsung s8 HS Code for Export & Import - Seair Exim Solutions
      https://www.seair.co.in/samsung-s8-hs-code.aspx
      Search Samsung s8 HS Code on Seair.co.in. Get verified Samsung s8 trade data with shipment records, exporter-importer de
  [2] Turkey Imports – Samsung galaxy s8 plus ( HS Code 851713000000)
      https://eximtradedata.com/search/country-turkey/type-import/product-samsung-galaxy-s8-plus/hscode-851713000000
      Access 2025 Turkey import data for Samsung galaxy s8 plus ( HS Code 851713000000) with verifie

---
## 5. Jina AI Reader  
Requires `JINA_API_KEY` in `.env` (free tier available at https://jina.ai/reader).  
Fetches and parses a URL into clean Markdown.

In [5]:
from search.providers.jina import JinaProvider

JINA_KEY = os.getenv("JINA_API_KEY")
if not JINA_KEY:
    print("⚠️  JINA_API_KEY not set – skipping.")
else:
    provider = JinaProvider(api_key=JINA_KEY)
    result = provider.search("HS code Samsung Galaxy S8", num=5)
    show(result)

⚠️  JINA_API_KEY not set – skipping.


---
## 6. Brave Search  
Requires `BRAVE_API_KEY` in `.env` (free plan: https://api.search.brave.com).

In [ ]:
from search.providers.brave import BraveProvider

BRAVE_KEY = os.getenv("BRAVE_API_KEY")
if not BRAVE_KEY:
    print("⚠️  BRAVE_API_KEY not set – skipping.")
else:
    provider = BraveProvider(api_key=BRAVE_KEY)
    result = provider.search("HS code Samsung Galaxy S8 smartphone", count=5)
    show(result)

---
## 7. SearXNG  
Self-hosted, no API key.  
Set `SEARXNG_BASE_URL` in `.env` or pass it directly (e.g. `http://localhost:8080`).

In [ ]:
from search.providers.searxng import SearXNGProvider

SEARXNG_URL = os.getenv("SEARXNG_BASE_URL", "")
if not SEARXNG_URL:
    print("⚠️  SEARXNG_BASE_URL not set – skipping.")
else:
    provider = SearXNGProvider()
    result = provider.search("HS code Samsung Galaxy S8", base_url=SEARXNG_URL, num_results=5)
    show(result)

---
## 8. WebCrawler (Playwright)  
No API key required. Crawls a URL and returns all visible text.  
Target: Chinese HS-code lookup page for "烘干机" (dryer).

In [ ]:
import concurrent.futures
from search.providers.webcrawler import WebCrawlerProvider

URL = "https://www.i5a6.com/hscode/key/%E7%83%98%E5%B9%B2%E6%9C%BA"

# Playwright sync_api conflicts with Jupyter's running event loop.
# Run the crawl in a thread to avoid that limitation.
provider = WebCrawlerProvider()

def _crawl():
    return provider.search(URL)

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as exe:
    result = exe.submit(_crawl).result(timeout=60)

print(f"provider : {result.provider}")
print(f"url      : {URL}")
print(f"crawled  : {result.metadata.get('crawl_success')}")
print(f"\n── First 1000 chars of page text ──")
print(result.answer[:1000])


### 8b. WebCrawler — with CSS selector  
Extract only the main results table from the same page.

In [ ]:
def _crawl_sel():
    return provider.search(URL, selector="table, .result, main, article")

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as exe:
    result_sel = exe.submit(_crawl_sel).result(timeout=60)

print(f"crawled (selector) : {result_sel.metadata.get('crawl_success')}")
print(result_sel.answer[:1000])
